In [ ]:
import cedalion.dot
from pathlib import Path
import trimesh
import pymeshlab
import pyvista as pv
import cedalion.vis.blocks as vbx
import numpy as np
import nibabel.freesurfer
import nibabel
from collections import Counter
import matplotlib.pyplot as p
import cedalion.dataclasses as cdc
import cedalion.geometry.segmentation as segm
import cedalion.geometry.meshing as meshing
from cedalion.io.anatomy import trimesh_from_freesurfer
from tqdm import tqdm
from scipy.spatial import KDTree
import scipy.optimize
import scipy.spatial.distance
import xarray as xr
import scipy.spatial
import pandas as pd
import re
from scipy.sparse import coo_array
import json

## Colin 27

In [ ]:
head_ijk = cedalion.dot.get_standard_headmodel("colin27")
head_ras = head_ijk.apply_transform(head_ijk.t_ijk2ras)

FSDIR = Path("/home/eike/Projekte/ibslab/30_dev/data/cedalion_datasets/fs_reconall_colin27")

lh_labels, lh_colors, lh_labelnames = nibabel.freesurfer.read_annot(
    FSDIR / "label/lh.Schaefer2018_600Parcels_17Networks_order.annot",
    orig_ids=False,
)
rh_labels, rh_colors, rh_labelnames = nibabel.freesurfer.read_annot(
    FSDIR / "label/rh.Schaefer2018_600Parcels_17Networks_order.annot",
    orig_ids=False,
)

# labels are indices into the lh_labelnames and rh_labelnames arrays.
# concatenate and add an offset to rh_labels
all_labels = np.concat((lh_labels, rh_labels + len(lh_labelnames)))
all_labelnames = np.concat((lh_labelnames, rh_labelnames))
all_colors = np.vstack((lh_colors, rh_colors))

m320k = trimesh.load_mesh(
    FSDIR / "ibs/cortex_pial_high.obj"
)


In [ ]:
def fs_to_shakiba(l):
    if match := re.match("17Networks_([L,R]H)_(.+)", l):
        return f"{match.group(2)}_{match.group(1)}"
    else:
        return l

mapping_fs = {i: l.decode("utf8") for i, l in enumerate(all_labelnames)}
mapping_fs_trafo = {k: fs_to_shakiba(v) for k, v in mapping_fs.items()}
colors_fs_trafo = {v: all_colors[k][:3] for k, v in mapping_fs_trafo.items()}
print(mapping_fs)
print(mapping_fs_trafo)
print(colors_fs_trafo)

In [ ]:
vol_labels = (
    nibabel.load(
        FSDIR / "ibs/cedalion_parcellation_600Parcels_17Networks_colin27.nii.gz"
    )
    .get_fdata()
    .astype(int)
)
print(np.unique(vol_labels))
print(vol_labels.shape)

In [ ]:
df = pd.read_csv(
    FSDIR / "ibs/cedalion_parcellation_600Parcels_17Networks_colin27_labels.csv",
    sep="\t",
    header=None,
)
mapping_nils = {0: "Background"} | {r[0]: r[1] for _, r in df.iterrows()}
mapping_nils

In [ ]:
brain_mask = head_ijk.segmentation_masks.sel(segmentation_type=["gm", "wm"]).any(
    "segmentation_type"
)

cell_indices = np.flatnonzero(brain_mask)

cell_coords = segm.cell_coordinates(brain_mask, flat=True).astype(float)
cell_coords += 0.5 # cell centers

cell_coords = cell_coords.assign_coords(
    {"parcel": ("label", np.asarray([mapping_nils[i] for i in vol_labels.flatten()]))}
)

cell_coords

In [ ]:
fssurf = cedalion.dataclasses.TrimeshSurface(
    m320k,
    crs=head_ijk.crs,
    units=cedalion.units(None).units,
    vertex_coords={"parcel": np.asarray([mapping_fs_trafo[i] for i in all_labels])},
)
print(fssurf.units)
fssurf.vertices

In [ ]:
inflmesh = trimesh_from_freesurfer(
    FSDIR / "surf/lh.inflated",
    FSDIR / "surf/rh.inflated",
    crs=fssurf.crs,
    units=cedalion.units(None).units,
)
inflmesh

In [ ]:
plt = pv.Plotter()
vbx.plot_surface(plt, head_ijk.scalp, color="w", opacity=0.3)

vbx.plot_surface(
    plt,
    fssurf,
    color=[colors_fs_trafo[i] for i in fssurf.vertices.parcel.values],
    opacity=1.0,
)

plt.show()

In [ ]:
plt = pv.Plotter()
vbx.plot_surface(
    plt,
    inflmesh,
    color=[colors_fs_trafo[i] for i in fssurf.vertices.parcel.values],
    opacity=1.0,
)
plt.show()

In [ ]:
tmp_surf = fssurf
quality = None
selected = False
voxel_stealing = False
#for nvertices in [30000, 28000, 27000, 26000, 25000]:
for nvertices in [25500, 25000]:
    if nvertices <= 26000:
        voxel_stealing = True

    print(f"{tmp_surf.nvertices} -> {nvertices}")
    tmp_surf = meshing.decimate_mesh(
        tmp_surf, nvertices, quality, selected, selection_threshold=0.5
    )

    # v2v, voxel_count = map_voxels_to_vertices(tmp_surf, cell_coords)
    v2v, voxel_count = meshing.parcel_aware_voxels_to_vertices_map(
        tmp_surf,
        cell_coords,
        voxel_stealing=voxel_stealing,
        skip_parcels=["Background"],
    )

    bad_vertices_mask = (voxel_count == 0) & ~(
        tmp_surf.vertices.parcel.str.startswith("Background").values
    )
    nprobvoxels = np.sum(bad_vertices_mask)
    print(f"problematic vertices: {nprobvoxels}")

    t = KDTree(tmp_surf.vertices.pint.dequantify()[bad_vertices_mask])
    dists, indices = t.query(tmp_surf.vertices.pint.dequantify())

    quality = np.clip(dists / 15, 0, 1.0)

    selected = False

reduced_surf_ijk = tmp_surf
reduced_surf_ijk

In [ ]:
plt = pv.Plotter()
vbx.plot_surface(plt, reduced_surf_ijk, color=voxel_count == 0, opacity=1)
plt.show()

In [ ]:
voxel_count_upscaled = meshing.upscale_scalars(
    fssurf, reduced_surf_ijk, np.clip(voxel_count, 0, 1)
)

plt = pv.Plotter()
vbx.plot_surface(plt, inflmesh, color=voxel_count_upscaled, opacity=1)
plt.show()

In [ ]:
voxel_indices = np.nonzero((v2v >= 0).values)[0]
vertex_indices = v2v.values[voxel_indices]

print(voxel_indices)
print(vertex_indices)
assert len(voxel_indices) == len(vertex_indices)


map_voxel_to_vertex = coo_array(
    (np.ones(len(voxel_indices)), (voxel_indices, vertex_indices)),
    shape=(len(v2v), tmp_surf.nvertices),
)

map_voxel_to_vertex

In [ ]:
colin_mni305 = reduced_surf_ijk.apply_transform(head_ijk.t_ijk2ras)
colin_mni305.crs = "mni305"
colin_mni305

In [ ]:
colin_mni152 = colin_mni305.apply_transform(cedalion.dot.utils.mni305_to_mni152)

In [ ]:
dists, vertex_ids = fssurf.kdtree.query(tmp_surf.mesh.vertices)

colin_mni305.vertex_coords["mni152_r"] = colin_mni152.vertices[:,0].data
colin_mni305.vertex_coords["mni152_a"] = colin_mni152.vertices[:,1].data
colin_mni305.vertex_coords["mni152_s"] = colin_mni152.vertices[:,2].data
colin_mni305.vertex_coords["fsaverage_vertex_id"] = vertex_ids

In [ ]:
colin_mni305.vertices

In [ ]:
OUTDIR = Path("/home/eike/Projekte/ibslab/30_dev/data/cedalion_datasets/hm_colin27")

In [ ]:
df_vertex = pd.DataFrame(
    {"vertex" : colin_mni305.vertices.label.values,
     "fsaverage_vertex" : colin_mni305.vertices.fsaverage_vertex_id.values,
     "parcel" : colin_mni305.vertices.parcel.values,
     "mni152_r" : colin_mni305.vertices.mni152_r.values,
     "mni152_a" : colin_mni305.vertices.mni152_a.values,
     "mni152_s" : colin_mni305.vertices.mni152_s.values,
     }
)
df_vertex

In [ ]:
df_vertex.to_csv(OUTDIR / "brain_vertex_coordinates.csv", index=False, float_format="%.3f")

In [ ]:
with ( OUTDIR/ "mask_brain.obj").open("w") as fout:
    fout.write(trimesh.exchange.obj.export_obj(reduced_surf_ijk.mesh, header="Colin27 cortex mesh"))



In [ ]:
parcel_colors = {k : list(map(int, v)) for k,v in colors_fs_trafo.items() if k != "Background+FreeSurfer_Defined_Medial_Wall"}
with (OUTDIR / "parcel_colors.json").open("w") as fout:
    json.dump(parcel_colors, fout, indent=2)

In [ ]:
scipy.sparse.save_npz(OUTDIR/ "voxel_to_vertex_brain.npz", map_voxel_to_vertex)

In [ ]:
scipy.sparse.load_npz(OUTDIR/ "voxel_to_vertex_brain.npz")

In [ ]:
head_ijk.voxel_to_vertex_brain